In [1]:
import warnings
warnings.filterwarnings("ignore")
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np


Instructions for updating:
non-resource variables are not supported in the long term


In [2]:
# 학습 데이터: [공부시간, 과외시간], 7행 2열
x = [[2, 3], [4, 3], [6, 4], [8, 6], [10, 7], [12, 8], [14, 9]]
print(x)
print(type(x))

xData = np.array(x)
print(xData)
print(type(xData))

[[2, 3], [4, 3], [6, 4], [8, 6], [10, 7], [12, 8], [14, 9]]
<class 'list'>
[[ 2  3]
 [ 4  3]
 [ 6  4]
 [ 8  6]
 [10  7]
 [12  8]
 [14  9]]
<class 'numpy.ndarray'>


In [3]:
# 실제값(레이블): [합격여부], 1행 7열
y = [0, 0, 0, 1, 1, 1, 1]
print(y)
print(type(y))

yData = np.array(y)
print(yData)
print(type(yData))

# reshape() 메소드는 numpy에서 데이터는 그대로 유지한채 배열의 형태(차원)를 변경한다. 1행 7열 nump y배열 => 7행 1열 numpy 배열
yData = np.array(y).reshape(7, 1)
print(yData)
print(type(yData))

[0, 0, 0, 1, 1, 1, 1]
<class 'list'>
[0 0 0 1 1 1 1]
<class 'numpy.ndarray'>
[[0]
 [0]
 [0]
 [1]
 [1]
 [1]
 [1]]
<class 'numpy.ndarray'>


xData와 yData를 저장할 placeholder를 만든다.

In [4]:
# placeholder에 2차원 이상의 numpy 배열을 대입하는 경우 shape 속성을 이용해서 대입될 2차원 이상의 배열의 차원을 지정해야 한다.
# [None, 2] => placeholder에 대입될 배열의 행의 개수는 몇개라도 상관없고 열의 개수는 무조건 2개라는 의미이다.
X = tf.placeholder(dtype=tf.float64, shape=[None, 2]) # xData가 대입될 placeholder를 선언한다.
Y = tf.placeholder(dtype=tf.float64, shape=[None, 1]) # yData가 대입될 placeholder를 선언한다.

가중치와 바이어스의 값을 임의의 값으로 정한다.

In [5]:
a = tf.Variable(tf.random_uniform([2, 1], dtype=tf.float64)) # 행렬의 내적을 이용하기 위해서 가중치는 난수를 2행 1열로 발생시킨다.
b = tf.Variable(tf.random_uniform([1], dtype=tf.float64))

sess = tf.Session()
sess.run(tf.global_variables_initializer())

print('a:\n', sess.run(a), sep='')
print('b:\n', sess.run(b), sep='')
print(f'a1 = {sess.run(a)[0][0]}, a2 = {sess.run(a)[1][0]}, b = {sess.run(b)[0]}')

a:
[[0.10118518]
 [0.42665725]]
b:
[0.64137059]
a1 = 0.10118518276093202, a2 = 0.42665725162548584, b = 0.6413705922531645


시그모이드 방정식, 오차 함수, 경사 하강법

In [6]:
# Y = 1 / (1 + np.e ** -(a * x + b))
# matmul() 메소드로 tensorflow에서 행렬의 내적을 계산한다.
# sigmoid() 메소드로 tensorflow에서 시그모이드 방정식을 계산한다.
y = tf.sigmoid(tf.matmul(X, a) + b) # 시그모이드 방정식, 예측값

# -tf.reduce_mean(실제값 * tf.log(예측값) + (1 - 실제값) * tf.log(1 - 예측값))
loss = -tf.reduce_mean(Y * tf.log(y) + (1 - Y) * tf.log(1 - y)) # 오차 함수

gradient_descent = tf.train.GradientDescentOptimizer(0.1).minimize(loss) # 경사 하강법

sigmoid() 메소드를 실행한 결과로 예측값(0 또는 1)을 계산한다.  
sigmoid() 메소드의 실행 결과가 0.5 이상이면 1을 0.5 미만이면 0을 리턴시킨다.

In [7]:
# cast() 메소드로 tensorflow에서 데이터 형변환을 할 수 있다.
# cast(형변환할 데이터, 형변환할 데이터 타입)
predict = tf.cast(tf.constant([1.9, 2.3]), dtype=tf.int32)
print(sess.run(predict))
# cast() 메소드는 형변환할 데이터가 boolena 타입일 경우 True는 1로 False는 0으로 형변환 한다.
predict = tf.cast(tf.constant([True, False]), dtype=tf.int32)
print(sess.run(predict))
predict = tf.cast(tf.constant([0.5 >= 0.5, 0.5 < 0.5]), dtype=tf.int32)
print(sess.run(predict))

[1 2]
[1 0]
[1 0]


In [8]:
# sigmoid() 메소드의 실행 결과(예측값, y)로 최종 예측값을 계산한다.
predict = tf.cast(y >= 0.5, dtype=tf.float64) # 최종 예측값

최종 예측값(predict)과 실제값(Y)이 일치하는 정도(정확도, accuracy)를 계산한다.

In [9]:
# equal() 메소드는 tensorflow에서 인수로 지정한 두 값이 같으면 True, 다르면 False를 리턴한다.
print(sess.run(tf.equal(1, 1)), sess.run(tf.equal(1, 0)))
# 정확도는 equal() 메소드도 예측값과 실제값이 같은가 비교 후 리턴되는 True 또는 False를 cast() 메소드로 1 또는 0로 형변환하고 평균을 계산하면 된다.
accuracy = tf.reduce_mean(tf.cast(tf.equal(predict, Y), dtype=tf.float64))

True False


학습시킨다.

In [10]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())

for epoch in range(10001):
    _, _loss, _a, _b, _predict, _accuracy = sess.run([gradient_descent, loss, a, b, predict, accuracy], feed_dict={X: xData, Y: yData})
    if epoch % 500 == 0:
        print('epoch: {:5d}, loss: {:7.3f}, a1: {:7.3f}, a2: {:7.3f}, b: {:7.3f}, accuracy: {:7.2%}'.format(
            epoch, _loss, _a[0][0], _a[1][0], _b[0], _accuracy
        ))

epoch:     0, loss:   1.593, a1:   0.517, a2:  -0.078, b:   0.700, accuracy:  57.14%
epoch:   500, loss:   0.224, a1:   0.924, a2:  -0.558, b:  -3.240, accuracy:  85.71%
epoch:  1000, loss:   0.146, a1:   0.755, a2:   0.029, b:  -5.097, accuracy: 100.00%
epoch:  1500, loss:   0.107, a1:   0.571, a2:   0.533, b:  -6.373, accuracy: 100.00%
epoch:  2000, loss:   0.084, a1:   0.417, a2:   0.938, b:  -7.348, accuracy: 100.00%
epoch:  2500, loss:   0.069, a1:   0.292, a2:   1.265, b:  -8.138, accuracy: 100.00%
epoch:  3000, loss:   0.059, a1:   0.192, a2:   1.536, b:  -8.802, accuracy: 100.00%
epoch:  3500, loss:   0.051, a1:   0.109, a2:   1.765, b:  -9.375, accuracy: 100.00%
epoch:  4000, loss:   0.045, a1:   0.039, a2:   1.961, b:  -9.878, accuracy: 100.00%
epoch:  4500, loss:   0.040, a1:  -0.020, a2:   2.133, b: -10.328, accuracy: 100.00%
epoch:  5000, loss:   0.037, a1:  -0.072, a2:   2.285, b: -10.734, accuracy: 100.00%
epoch:  5500, loss:   0.033, a1:  -0.117, a2:   2.421, b: -11.104

In [11]:
print('실제값\n', yData, sep='')
print('최종 예측값\n', _predict, sep='')
print('정확도: {:7.2%}'.format(_accuracy))

실제값
[[0]
 [0]
 [0]
 [1]
 [1]
 [1]
 [1]]
최종 예측값
[[0.]
 [0.]
 [0.]
 [1.]
 [1.]
 [1.]
 [1.]]
정확도: 100.00%


In [12]:
# testData = np.array([[5, 8]])
testData = np.array([5, 8]).reshape(1, 2)
print('pass' if sess.run(predict, feed_dict={X: testData})[0][0] == 1.0 else 'fail')

pass
